# OSCARS XAS Demonstrator — full pipeline

Find public **XAS** data at two facilities through their own discovery
mechanisms, get it into the **NXxas** NeXus format, run the **ewoks/est EXAFS**
workflow, and write results back as NOMAD entries — all from this notebook,
running in **NORTH Jupyter** on `oasis-b`.

* **ESRF (ID21)** via the [`nomad-semantic-web-service`](https://github.com/FAIRmat-NFDI/nomad-semantic-web-service)
  package: PaNET→ESRFET ontology mapping + ICAT+ catalogue + anonymous IDS
  download → `pynxtools-xas` conversion to NXxas.
* **BESSY II** via the NOMAD search API, filtering by the pynxtools **NXxas
  definition** (already NXxas — no conversion).
* **Common tail** — the ewoks/est EXAFS workflow → result entries.

The companion notebook `2_eln_esrf_plus_bessy.ipynb` shows the variant where the
ESRF data is pulled **inside NOMAD** via the plugin's ELN instead of here.

## Setup

Configuration lives in the constants at the top of the next cell (BESSY
endpoint, upload id, ...) — defaults work unchanged both locally and in NORTH
Jupyter on `oasis-b`. The only thing actually environment-driven is
`NOMAD_SWS_SRC`, a dev-tree escape hatch to point at a local checkout of
`nomad-semantic-web-service` instead of an installed package. Downloaded raw
files and results are written **into this upload folder** (`downloads/`,
`results/`), so NOMAD parses them into their own entries — no separate upload
step.

The ewoks graph runs headless, so `QT_QPA_PLATFORM=offscreen` is set before any
Orange import.

In [ ]:
import os, sys, subprocess
from datetime import date
from pathlib import Path

os.environ["QT_QPA_PLATFORM"] = "offscreen"   # ewoksorange pulls in Orange GUI imports

# In a dev tree, point at the semantic-web-service source instead of installing.
_sws_src = os.environ.get("NOMAD_SWS_SRC")
if _sws_src and _sws_src not in sys.path:
    sys.path.insert(0, _sws_src)


def _ensure(mods, pip_args):
    "pip-install pip_args only if any of mods is missing (skipped in a prepared kernel)."
    missing = [m for m in mods if __import__("importlib").util.find_spec(m) is None]
    if missing:
        print("installing:", " ".join(pip_args))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pip_args], check=True)


_ensure(["ewoks", "ewoksorange", "est"], ["ewoks", "ewoksorange", "est", "PyMca5", "PyQt5"])
_ensure(["requests", "yaml"], ["requests", "pyyaml"])
_ensure(["nomad_semantic_web_service"], ["nomad-semantic-web-service"])
_ensure(["jupyterlab_h5web"], ["jupyterlab-h5web"])

sys.path.insert(0, str(Path.cwd()))   # so `import oscars_demo` works in NORTH
import oscars_demo as od

PYNX = "pynx"                          # pynxtools-xas CLI
BESSY_API = od.NOMAD_STAGING           # switch to od.NOMAD_BESSY for the live oasis
BESSY_UPLOAD = "zMg1PGypQRa4yA05cyM8Pw"

DOWNLOADS = Path("downloads"); DOWNLOADS.mkdir(exist_ok=True)
RESULTS   = Path("results");   RESULTS.mkdir(exist_ok=True)
print("ready.")

## ESRF ID21 — semantic discovery + download + convert

**Semantic step first:** map a PaNET technique term to the facility's ESRFET
vocabulary through the ontology (`owl:equivalentClass`). This is the
findable-via-semantics contribution — the same mapping the plugin's ELN and REST
route perform.

In [ ]:
mapping = od.esrf_map_technique("PaNET01196", vocabulary="PANET")   # PaNET "XAS"
print("PaNET01196  ->", mapping["resolved_iri"], f"({mapping['relation']})")

In [ ]:
# ICAT+ discovery. ID21's public datasets are annotated with the XAS technique,
# so the resolved ESRFET XAS IRI filters them server-side; they are kept on disk
# (unlike ESRF's tape-archived EXAFS beamlines). See ESRF_ICAT.md.
esrf = od.esrf_search_datasets(
    date(2021, 1, 1), date(2022, 12, 31),
    beamline="ID21", technique_pids=mapping["resolved_iri"], limit=5,
)
status = od.esrf_dataset_status([d["id"] for d in esrf])
for d in esrf:
    print(f"{d['id']}  {d.get('sampleName',''):<18} {status.get(d['id'], 'UNKNOWN')}")

In [ ]:
# Download the newest dataset's .h5 file(s) (anonymous IDS). A dataset may hold
# more than one .h5 (e.g. the rich Bliss scan plus a sparse metadata sibling).
target = esrf[0]
esrf_h5 = od.esrf_download_h5(target["id"], DOWNLOADS / "esrf",
                              sample_name=target.get("sampleName"))
print("ESRF .h5:", [p.name for p in esrf_h5])

In [ ]:
# Convert *every* downloaded ESRF .h5 to NXxas with pynxtools-xas (fluorescence-
# yield XANES; element/edge inferred from the active emission line where the
# layout provides it). Files no parser recognizes are skipped.
esrf_nxs = od.convert_all_to_nxxas(esrf_h5, DOWNLOADS / "esrf", nxdl="NXxas", pynx=PYNX)
print("ESRF NXxas:", [p.name for p in esrf_nxs])

### View one downloaded file with H5Web

H5Web renders any HDF5/NeXus file inline. Point it at one of the converted
NXxas files to browse the spectra.

In [ ]:
from jupyterlab_h5web import H5Web

H5Web(esrf_nxs[0] if esrf_nxs else print("no file to show"))

## BESSY II — discover NXxas via the NOMAD search API

BESSY data already lives in a NOMAD oasis as **NXxas `.nxs` entries**. We search
*as if we didn't know where it was*, by the pynxtools NeXus **definition**
(`definition == NXxas`) — the facility-agnostic discovery contribution — then
download the raw `.nxs` (already NXxas, no conversion needed).

In [ ]:
bessy_hits = od.search_nxxas_entries(
    nomad_api=BESSY_API,
    definition="NXxas",
    upload_id=BESSY_UPLOAD,
)
print(f"BESSY: {len(bessy_hits)} NXxas entries")
bessy_nxs = []
for hit in bessy_hits:
    # Tolerate a per-entry download hiccup (e.g. a transient rate limit) so one
    # bad entry doesn't abort discovery of the rest.
    try:
        got = od.download_nxs_entry(
            hit["entry_id"], DOWNLOADS / "bessy", nomad_api=hit["nomad_api"]
        )
        bessy_nxs.extend(got)
    except Exception as exc:
        print(f"   [skip] {hit['entry_id']}: {type(exc).__name__}: {exc}")
print("BESSY .nxs:", [p.name for p in bessy_nxs])

## Common tail — ewoks/est EXAFS workflow + result entries

Every discovered `.nxs` runs through the **same** workflow (Input → Normalization
→ EXAFS → k-weight → Fourier transform → Output), driven headlessly by
`run_workflow.py`. Base-`NXxas` files (BESSY foils and the ESRF ID21
fluorescence files alike) carry the signal at `entry/intensity`, so we pass
`signal="intensity"`; `run_ewoks_exafs` also strips non-physical "a.u." unit
tags that est/PyMca's pint would misread. One `.archive.yaml` per result is
written into this upload, so each result becomes its own NOMAD entry.

In [ ]:
# Every converted/discovered NXxas file is now an entry in this upload. Run the
# EXAFS workflow on a representative subset (a per-facility cap) to keep the demo
# quick; raise this constant to process more.
EWOKS_MAX = 2
jobs = (
    [("ESRF@ID21", p) for p in esrf_nxs[:EWOKS_MAX]]
    + [("BESSY", p) for p in bessy_nxs[:EWOKS_MAX]]
)
print("ewoks jobs:", [(f, Path(p).name) for f, p in jobs])

In [ ]:
results = []
for facility, nxs in jobs:
    out_h5 = RESULTS / f"{Path(nxs).stem}_result.h5"
    try:
        res = od.run_ewoks_exafs(nxs, out_h5, signal="intensity")  # energy_unit auto-detected from the file
        results.append((facility, Path(nxs).stem, nxs, res["result_file"]))
        print(f"[OK]   {facility:<12} {Path(nxs).stem}: {Path(res['result_file']).name}")
    except Exception as exc:
        print(f"[FAIL] {facility:<12} {Path(nxs).stem}: {type(exc).__name__}: {exc}")

for facility, sample, nxs, result_file in results:
    od.write_result_entry(
        RESULTS / f"{Path(nxs).stem}.archive.yaml",
        source_facility=facility, source_id=sample,
        nxs_file=str(nxs), result_file=str(result_file), sample_name=sample,
    )
print(f"\nwrote {len(results)} result entr{'y' if len(results)==1 else 'ies'} into {RESULTS}/")

### View a result with H5Web

Point H5Web at one of the processed result files to browse the data.

In [ ]:
from jupyterlab_h5web import H5Web

H5Web(str(results[0][3])) if results else print("no results to show")